# Basics of ML

**Q: Perform Regression and Decision Tree**

In [3]:
import pandas as pd

df = pd.read_csv('D:/CitrusBits/pythonic-rebirth/datasets/insurance_ml.csv')

# df.describe()
# df.info()
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])  # male/female → 0/1
df['smoker'] = le.fit_transform(df['smoker'])  # yes/no → 0/1

# OneHotEncoder for region (no natural order, more than 2 categories)
if 'region' in df.columns:
    onehot_encoder = OneHotEncoder(sparse_output=False)
    region_encoded = onehot_encoder.fit_transform(df[['region']])

    region_encoded_df = pd.DataFrame(
        region_encoded,
        columns=onehot_encoder.get_feature_names_out(['region']),
        index=df.index
    )

    df = df.drop('region', axis=1)
    df = pd.concat([df, region_encoded_df], axis=1)

df.head()

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0.0,0.0,0.0,1.0
1,18,1,33.770,1,0,1725.55230,0.0,0.0,1.0,0.0
2,28,1,33.000,3,0,4449.46200,0.0,0.0,1.0,0.0
3,33,1,22.705,0,0,21984.47061,0.0,1.0,0.0,0.0
4,32,1,28.880,0,0,3866.85520,0.0,1.0,0.0,0.0


Why scale these three specifically: age, bmi, and children are your remaining raw numeric features with different ranges (age ~18-64, bmi ~15-55, children 0-5). sex and smoker don't need scaling since they're already 0/1 binary. The region_* one-hot columns also don't need scaling as they're already 0/1. And charges stays untouched since it's our target, and not a feature.

In [5]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['age', 'bmi', 'children']

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

df.head()

,age,sex,bmi,children,smoker,charges,region_northeast,region_northwest,region_southeast,region_southwest
0,-1.438764,0,-0.453320,-0.908614,1,16884.92400,0.0,0.0,0.0,1.0
1,-1.509965,1,0.509621,-0.078767,0,1725.55230,0.0,0.0,1.0,0.0
2,-0.797954,1,0.383307,1.580926,0,4449.46200,0.0,0.0,1.0,0.0
3,-0.441948,1,-1.305531,-0.908614,0,21984.47061,0.0,1.0,0.0,0.0
4,-0.513149,1,-0.292556,-0.908614,0,3866.85520,0.0,1.0,0.0,0.0


In [6]:
from sklearn.model_selection import train_test_split

X = df.drop('charges', axis=1)
y = df['charges']

# Step 1: split off test set (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Step 2: split remaining 80% into train (70% of total) and validation (10% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.10 / 0.80), random_state=42
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (936, 9)
X_val shape: (134, 9)
X_test shape: (268, 9)


In [7]:
# Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_val_preds = lr_model.predict(X_val)

lr_val_mse = mean_squared_error(y_val, lr_val_preds)
lr_val_r2 = r2_score(y_val, lr_val_preds)

print(f"Linear Regression (Validation) → MSE: {lr_val_mse:.2f}, R²: {lr_val_r2:.4f}")

Linear Regression (Validation) → MSE: 40495574.19, R²: 0.7778


In [8]:
# Decision Tree

from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)

dt_val_preds = dt_model.predict(X_val)

dt_val_mse = mean_squared_error(y_val, dt_val_preds)
dt_val_r2 = r2_score(y_val, dt_val_preds)

print(f"Decision Tree (Validation) → MSE: {dt_val_mse:.2f}, R²: {dt_val_r2:.4f}")

Decision Tree (Validation) → MSE: 26989461.41, R²: 0.8519


In [9]:
print(f"{'Model':<20}{'MSE':<15}{'R²':<10}")
print(f"{'Linear Regression':<20}{lr_val_mse:<15.2f}{lr_val_r2:<10.4f}")
print(f"{'Decision Tree':<20}{dt_val_mse:<15.2f}{dt_val_r2:<10.4f}")

Model               MSE            R²        
Linear Regression   40495574.19    0.7778    
Decision Tree       26989461.41    0.8519    


In [10]:
import numpy as np

print(f"Linear Regression RMSE: ${np.sqrt(40495574.19):.2f}")
print(f"Decision Tree RMSE: ${np.sqrt(26989461.41):.2f}")

Linear Regression RMSE: $6363.61
Decision Tree RMSE: $5195.14


In [11]:
import joblib
import os

model_dir = "D:/CitrusBits/pythonic-rebirth/models"
os.makedirs(model_dir, exist_ok=True)

lr_path = os.path.join(model_dir, "linear_regression_model.pkl")
dt_path = os.path.join(model_dir, "decision_tree_model.pkl")

# Linear Regression
if os.path.exists(lr_path):
    print("Loading existing Linear Regression model...")
    lr_model = joblib.load(lr_path)
else:
    print("Training new Linear Regression model...")
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    joblib.dump(lr_model, lr_path)
    print("Model saved.")

# Decision Tree
if os.path.exists(dt_path):
    print("Loading existing Decision Tree model...")
    dt_model = joblib.load(dt_path)
else:
    print("Training new Decision Tree model...")
    dt_model = DecisionTreeRegressor(random_state=42, max_depth=5)
    dt_model.fit(X_train, y_train)
    joblib.dump(dt_model, dt_path)
    print("Model saved.")

Training new Linear Regression model...
Model saved.
Training new Decision Tree model...
Model saved.
